# 18uA — Chronological partition and expanding validation folds

This stage freezes the date partition before any corrected
residual model, Gaussian process, tree model, calibration map or
trading threshold is fitted.

The final date blocks are:

- **history-only warm-up:** 16 March–11 April 2026;
- **development:** 12 April–21 May 2026;
- **locked internal holdout:** 22–31 May 2026;
- **external out-of-time test:** 1–30 June 2026.

A date–decision observation is model-ready only when its current
deterministic forecast path is available and it has at least
sixteen publication-admissible historical residuals from the same
decision rule. The threshold of sixteen preserves the declared
minimum initial training window.

The 38 development dates are divided into four contiguous
expanding-origin validation blocks of 10, 10, 9 and 9 settlement
dates. Every decision rule and every contract sharing a settlement
date inherits the same date partition and fold.

This notebook assigns the protocol. It does not select a model,
feature family, kernel, hyperparameter, calibrator or trading rule.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18uA"
MIN_HISTORY = 16
CONTRACTS_PER_BOOK = 11

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]
RULE_ORDER = {
    rule: position
    for position, rule in enumerate(RULES)
}

WARMUP_END = pd.Timestamp("2026-04-11")
DEVELOPMENT_START = pd.Timestamp("2026-04-12")
DEVELOPMENT_END = pd.Timestamp("2026-05-21")
HOLDOUT_START = pd.Timestamp("2026-05-22")
HOLDOUT_END = pd.Timestamp("2026-05-31")
EXTERNAL_START = pd.Timestamp("2026-06-01")
EXTERNAL_END = pd.Timestamp("2026-06-30")

A_DIR = (
    ROOT
    / "data/processed/18tA_hko_publication_availability"
)
B_DIR = (
    ROOT
    / "data/processed/18tB_admissible_historical_information_sets"
)

AVAILABILITY_PATH = (
    A_DIR / "18tA_hko_publication_availability_panel.csv"
)
A_SUMMARY_PATH = A_DIR / "18tA_summary.json"
A_MANIFEST_PATH = A_DIR / "18tA_sha256_manifest.csv"

DECISION_PATH = B_DIR / "18tB_decision_inventory.csv"
INFORMATION_PATH = (
    B_DIR / "18tB_information_set_summary.csv"
)
B_SUMMARY_PATH = B_DIR / "18tB_summary.json"
B_MANIFEST_PATH = B_DIR / "18tB_sha256_manifest.csv"

OUT_DIR = (
    ROOT
    / "data/processed/18uA_chronological_partition_and_folds"
)
REPORT_DIR = (
    ROOT
    / "reports/18uA_chronological_partition_and_folds"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    AVAILABILITY_PATH,
    A_SUMMARY_PATH,
    A_MANIFEST_PATH,
    DECISION_PATH,
    INFORMATION_PATH,
    B_SUMMARY_PATH,
    B_MANIFEST_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified 18t input is missing: {path}"
        )

EXPECTED_BLOCKS = {
    "HISTORY_ONLY_WARMUP": {
        "dates": 25,
        "date_rule_rows": 100,
        "prediction_eligible_rows": 64,
        "model_ready_rows": 0,
        "contract_cells": 0,
    },
    "DEVELOPMENT": {
        "dates": 38,
        "date_rule_rows": 152,
        "prediction_eligible_rows": 152,
        "model_ready_rows": 144,
        "contract_cells": 1584,
    },
    "INTERNAL_HOLDOUT": {
        "dates": 10,
        "date_rule_rows": 40,
        "prediction_eligible_rows": 40,
        "model_ready_rows": 40,
        "contract_cells": 440,
    },
    "EXTERNAL_TEST": {
        "dates": 30,
        "date_rule_rows": 120,
        "prediction_eligible_rows": 119,
        "model_ready_rows": 119,
        "contract_cells": 1309,
    },
}

EXPECTED_FOLDS = {
    1: {
        "dates": 10,
        "start": "2026-04-12",
        "end": "2026-04-21",
        "validation_rows": 32,
        "contract_cells": 352,
    },
    2: {
        "dates": 10,
        "start": "2026-04-22",
        "end": "2026-05-01",
        "validation_rows": 40,
        "contract_cells": 440,
    },
    3: {
        "dates": 9,
        "start": "2026-05-02",
        "end": "2026-05-10",
        "validation_rows": 36,
        "contract_cells": 396,
    },
    4: {
        "dates": 9,
        "start": "2026-05-11",
        "end": "2026-05-21",
        "validation_rows": 36,
        "contract_cells": 396,
    },
}

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def verify_manifest(path: Path) -> None:
    manifest = pd.read_csv(path)
    failures: list[str] = []

    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path

        if not candidate.is_file():
            failures.append(f"MISSING: {row.path}")
            continue

        digest = sha256_file(candidate)
        if digest != row.sha256:
            failures.append(f"HASH: {row.path}")

        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f"SIZE: {row.path}")

    if failures:
        raise AssertionError(
            f"Manifest verification failed for {path}:\n"
            + "\n".join(failures)
        )


def evaluation_block(
    event_date: pd.Timestamp,
) -> str:
    date = pd.Timestamp(event_date).normalize()

    if date <= WARMUP_END:
        return "HISTORY_ONLY_WARMUP"
    if DEVELOPMENT_START <= date <= DEVELOPMENT_END:
        return "DEVELOPMENT"
    if HOLDOUT_START <= date <= HOLDOUT_END:
        return "INTERNAL_HOLDOUT"
    if EXTERNAL_START <= date <= EXTERNAL_END:
        return "EXTERNAL_TEST"

    raise AssertionError(
        f"Date outside frozen 18u universe: {date}"
    )


verify_manifest(A_MANIFEST_PATH)
verify_manifest(B_MANIFEST_PATH)

with A_SUMMARY_PATH.open(encoding="utf-8") as handle:
    a_summary = json.load(handle)
with B_SUMMARY_PATH.open(encoding="utf-8") as handle:
    b_summary = json.load(handle)

if a_summary.get("verdict") != "PASS":
    raise AssertionError("18tA is not a PASS release.")
if b_summary.get("verdict") != "PASS":
    raise AssertionError("18tB is not a PASS release.")

availability = pd.read_csv(
    AVAILABILITY_PATH,
    low_memory=False,
)
decisions = pd.read_csv(
    DECISION_PATH,
    low_memory=False,
)
information = pd.read_csv(
    INFORMATION_PATH,
    low_memory=False,
)

availability["event_date"] = pd.to_datetime(
    availability["event_date"],
    errors="raise",
)
availability[
    "hko_publication_available_utc"
] = pd.to_datetime(
    availability[
        "hko_publication_available_utc"
    ],
    utc=True,
    errors="raise",
)

decisions["event_date"] = pd.to_datetime(
    decisions["event_date"],
    errors="raise",
)
decisions["decision_cutoff_utc"] = pd.to_datetime(
    decisions["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
decisions["decision_cutoff_hkt"] = pd.to_datetime(
    decisions["decision_cutoff_hkt"],
    utc=True,
    errors="raise",
).dt.tz_convert("Asia/Hong_Kong")
decisions["current_prediction_eligible"] = parse_bool(
    decisions["current_prediction_eligible"],
    name="decisions.current_prediction_eligible",
)

information["current_event_date"] = pd.to_datetime(
    information["current_event_date"],
    errors="raise",
)
information[
    "current_decision_cutoff_utc"
] = pd.to_datetime(
    information[
        "current_decision_cutoff_utc"
    ],
    utc=True,
    errors="raise",
)
information[
    "current_prediction_eligible"
] = parse_bool(
    information["current_prediction_eligible"],
    name="information.current_prediction_eligible",
)

if len(availability) != 103:
    raise AssertionError(
        f"Expected 103 availability dates, found {len(availability)}"
    )
if len(decisions) != 412:
    raise AssertionError(
        f"Expected 412 decision rows, found {len(decisions)}"
    )
if len(information) != 412:
    raise AssertionError(
        f"Expected 412 information rows, found {len(information)}"
    )

print("Verified 18t inputs and manifests: PASS")

Verified 18t inputs and manifests: PASS


In [3]:
decision_key = [
    "event_date",
    "decision_rule",
]

info_columns = [
    "current_event_date",
    "current_decision_rule",
    "n_admissible_same_rule_residuals",
    "n_distinct_same_rule_history_dates",
    "earliest_same_rule_history_date",
    "latest_same_rule_history_date",
    "latest_same_rule_hko_availability_utc",
]

assignment = decisions.merge(
    information[info_columns].rename(
        columns={
            "current_event_date": "event_date",
            "current_decision_rule": "decision_rule",
        }
    ),
    on=decision_key,
    how="left",
    validate="one_to_one",
)

assignment = assignment.merge(
    availability[
        [
            "event_date",
            "availability_source",
            "hko_publication_available_utc",
        ]
    ].rename(
        columns={
            "availability_source": (
                "current_label_availability_source"
            ),
            "hko_publication_available_utc": (
                "current_label_available_utc"
            ),
        }
    ),
    on="event_date",
    how="left",
    validate="many_to_one",
)

assignment["evaluation_block"] = assignment[
    "event_date"
].map(evaluation_block)
assignment["minimum_same_rule_history"] = MIN_HISTORY
assignment["history_threshold_satisfied"] = (
    assignment[
        "n_admissible_same_rule_residuals"
    ]
    >= MIN_HISTORY
)
assignment["model_ready"] = (
    assignment["current_prediction_eligible"]
    & assignment["history_threshold_satisfied"]
    & ~assignment["evaluation_block"].eq(
        "HISTORY_ONLY_WARMUP"
    )
)
assignment["planned_contract_cells"] = np.where(
    assignment["model_ready"],
    CONTRACTS_PER_BOOK,
    0,
)
assignment["final_modelling_split_assigned"] = True
assignment["model_specification_selected"] = False
assignment["market_information_required_for_split"] = False
assignment["probability_bridge_required_for_split"] = False

development_dates = np.array(
    sorted(
        assignment.loc[
            assignment[
                "evaluation_block"
            ].eq("DEVELOPMENT"),
            "event_date",
        ].drop_duplicates()
    ),
    dtype="datetime64[ns]",
)

if len(development_dates) != 38:
    raise AssertionError(
        f"Expected 38 development dates, found {len(development_dates)}"
    )

fold_arrays = np.array_split(development_dates, 4)
fold_date_map: dict[pd.Timestamp, int] = {}

for fold_id, fold_dates in enumerate(
    fold_arrays,
    start=1,
):
    for date in fold_dates:
        fold_date_map[pd.Timestamp(date)] = fold_id

assignment["development_fold"] = (
    assignment["event_date"]
    .map(fold_date_map)
    .astype("Int64")
)

assignment["row_role"] = np.select(
    [
        assignment["evaluation_block"].eq(
            "HISTORY_ONLY_WARMUP"
        ),
        (
            assignment["evaluation_block"].eq(
                "DEVELOPMENT"
            )
            & assignment["model_ready"]
        ),
        (
            assignment["evaluation_block"].eq(
                "DEVELOPMENT"
            )
            & ~assignment["model_ready"]
        ),
        (
            assignment["evaluation_block"].eq(
                "INTERNAL_HOLDOUT"
            )
            & assignment["model_ready"]
        ),
        (
            assignment["evaluation_block"].eq(
                "INTERNAL_HOLDOUT"
            )
            & ~assignment["model_ready"]
        ),
        (
            assignment["evaluation_block"].eq(
                "EXTERNAL_TEST"
            )
            & assignment["model_ready"]
        ),
        (
            assignment["evaluation_block"].eq(
                "EXTERNAL_TEST"
            )
            & ~assignment["model_ready"]
        ),
    ],
    [
        "HISTORY_ONLY",
        "DEVELOPMENT_OOF_ELIGIBLE",
        "DEVELOPMENT_NOT_MODEL_READY",
        "LOCKED_HOLDOUT_ELIGIBLE",
        "LOCKED_HOLDOUT_NOT_MODEL_READY",
        "EXTERNAL_TEST_ELIGIBLE",
        "EXTERNAL_TEST_NOT_MODEL_READY",
    ],
    default="UNRESOLVED",
)

if assignment["row_role"].eq("UNRESOLVED").any():
    raise AssertionError(
        "At least one date-rule row has an unresolved role."
    )

print("Chronological row assignment: PASS")

Chronological row assignment: PASS


In [4]:
date_partition = (
    assignment.groupby(
        ["event_date", "evaluation_block"],
        as_index=False,
    )
    .agg(
        sample_block=("sample_block", "first"),
        date_rule_rows=("decision_rule", "size"),
        prediction_eligible_rules=(
            "current_prediction_eligible",
            "sum",
        ),
        model_ready_rules=("model_ready", "sum"),
        planned_contract_cells=(
            "planned_contract_cells",
            "sum",
        ),
        development_fold=(
            "development_fold",
            "first",
        ),
    )
    .sort_values("event_date")
    .reset_index(drop=True)
)

date_partition["all_four_rules_model_ready"] = (
    date_partition["model_ready_rules"].eq(4)
)
date_partition["any_rule_model_ready"] = (
    date_partition["model_ready_rules"].gt(0)
)
date_partition[
    "all_contracts_on_date_share_partition"
] = True
date_partition[
    "all_contracts_on_date_share_fold"
] = True
date_partition[
    "final_modelling_split_assigned"
] = True

if len(date_partition) != 103:
    raise AssertionError(
        f"Expected 103 date rows, found {len(date_partition)}"
    )

block_summary = (
    assignment.groupby(
        "evaluation_block",
        as_index=False,
    )
    .agg(
        settlement_dates=("event_date", "nunique"),
        date_rule_rows=("event_date", "size"),
        prediction_eligible_rows=(
            "current_prediction_eligible",
            "sum",
        ),
        model_ready_rows=("model_ready", "sum"),
        planned_contract_cells=(
            "planned_contract_cells",
            "sum",
        ),
    )
)

for block, expected in EXPECTED_BLOCKS.items():
    row = block_summary.loc[
        block_summary["evaluation_block"].eq(block)
    ]
    if len(row) != 1:
        raise AssertionError(
            f"Missing block summary row: {block}"
        )

    actual = {
        "dates": int(row["settlement_dates"].iloc[0]),
        "date_rule_rows": int(
            row["date_rule_rows"].iloc[0]
        ),
        "prediction_eligible_rows": int(
            row["prediction_eligible_rows"].iloc[0]
        ),
        "model_ready_rows": int(
            row["model_ready_rows"].iloc[0]
        ),
        "contract_cells": int(
            row["planned_contract_cells"].iloc[0]
        ),
    }

    if actual != expected:
        raise AssertionError(
            f"Block totals differ for {block}:\n"
            f"Expected={expected}\nActual={actual}"
        )

fold_rows: list[dict[str, Any]] = []
development_ready = assignment.loc[
    assignment["evaluation_block"].eq("DEVELOPMENT")
    & assignment["model_ready"]
].copy()

for fold_id, expected in EXPECTED_FOLDS.items():
    dates = date_partition.loc[
        date_partition["development_fold"].eq(
            fold_id
        ),
        "event_date",
    ].sort_values()
    rows = development_ready.loc[
        development_ready["development_fold"].eq(
            fold_id
        )
    ]

    actual = {
        "dates": int(dates.nunique()),
        "start": str(dates.min().date()),
        "end": str(dates.max().date()),
        "validation_rows": int(len(rows)),
        "contract_cells": int(
            rows["planned_contract_cells"].sum()
        ),
    }

    if actual != expected:
        raise AssertionError(
            f"Fold {fold_id} differs:\n"
            f"Expected={expected}\nActual={actual}"
        )

    fold_rows.append(
        {
            "development_fold": fold_id,
            "validation_start_date": dates.min(),
            "validation_end_date": dates.max(),
            "validation_dates": dates.nunique(),
            "validation_date_rule_rows": len(rows),
            "validation_contract_cells": int(
                rows["planned_contract_cells"].sum()
            ),
            "minimum_same_rule_history": MIN_HISTORY,
            "all_dates_contiguous_in_observed_sequence": True,
        }
    )

fold_summary = pd.DataFrame(fold_rows)

fold_rule_summary = (
    development_ready.groupby(
        [
            "development_fold",
            "decision_rule",
            "decision_rule_order",
        ],
        as_index=False,
    )
    .agg(
        validation_dates=("event_date", "nunique"),
        validation_date_rule_rows=("event_date", "size"),
        validation_contract_cells=(
            "planned_contract_cells",
            "sum",
        ),
        minimum_history_count=(
            "n_admissible_same_rule_residuals",
            "min",
        ),
        median_history_count=(
            "n_admissible_same_rule_residuals",
            "median",
        ),
        maximum_history_count=(
            "n_admissible_same_rule_residuals",
            "max",
        ),
    )
    .sort_values(
        ["development_fold", "decision_rule_order"]
    )
    .reset_index(drop=True)
)

if len(fold_rule_summary) != 16:
    raise AssertionError(
        f"Expected 16 fold-rule rows, found {len(fold_rule_summary)}"
    )

if not fold_rule_summary[
    "minimum_history_count"
].ge(MIN_HISTORY).all():
    raise AssertionError(
        "A development validation rule-fold starts below "
        "the minimum history threshold."
    )

print("Date, block and fold totals: PASS")
display(block_summary)
display(fold_summary)
display(fold_rule_summary)

Date, block and fold totals: PASS


,evaluation_block,settlement_dates,date_rule_rows,prediction_eligible_rows,model_ready_rows,planned_contract_cells
0,DEVELOPMENT,38,152,152,144,1584
1,EXTERNAL_TEST,30,120,119,119,1309
2,HISTORY_ONLY_WARMUP,25,100,64,0,0
3,INTERNAL_HOLDOUT,10,40,40,40,440


,development_fold,validation_start_date,validation_end_date,validation_dates,validation_date_rule_rows,validation_contract_cells,minimum_same_rule_history,all_dates_contiguous_in_observed_sequence
0,1,2026-04-12,2026-04-21,10,32,352,16,True
1,2,2026-04-22,2026-05-01,10,40,440,16,True
2,3,2026-05-02,2026-05-10,9,36,396,16,True
3,4,2026-05-11,2026-05-21,9,36,396,16,True


,development_fold,decision_rule,decision_rule_order,validation_dates,validation_date_rule_rows,validation_contract_cells,minimum_history_count,median_history_count,maximum_history_count
0,1,24h_prior,0,10,10,110,16,20.5,23
1,1,12h_prior,1,7,7,77,16,19.0,20
2,1,6h_prior,2,7,7,77,16,19.0,22
3,1,event_day_open,3,8,8,88,18,21.5,25
4,2,24h_prior,0,10,10,110,26,30.0,35
5,2,12h_prior,1,10,10,110,23,27.0,32
6,2,6h_prior,2,10,10,110,23,26.0,32
7,2,event_day_open,3,10,10,110,26,29.0,35
8,3,24h_prior,0,9,9,99,36,40.0,44
9,3,12h_prior,1,9,9,99,33,37.0,41


In [5]:
check_rows: list[dict[str, Any]] = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "verified_18t_inputs",
    (
        a_summary.get("verdict") == "PASS"
        and b_summary.get("verdict") == "PASS"
    ),
    "18tA and 18tB verdicts are PASS",
)
add_check(
    "date_partition_rows_103",
    len(date_partition) == 103,
    f"rows={len(date_partition)}",
)
add_check(
    "date_rule_rows_412",
    len(assignment) == 412,
    f"rows={len(assignment)}",
)
add_check(
    "minimum_history_frozen_at_16",
    MIN_HISTORY == 16,
    f"minimum={MIN_HISTORY}",
)
add_check(
    "development_dates_38",
    date_partition[
        "evaluation_block"
    ].eq("DEVELOPMENT").sum()
    == 38,
    "12 April through 21 May observed dates",
)
add_check(
    "holdout_dates_10",
    date_partition[
        "evaluation_block"
    ].eq("INTERNAL_HOLDOUT").sum()
    == 10,
    "22-31 May 2026",
)
add_check(
    "external_dates_30",
    date_partition[
        "evaluation_block"
    ].eq("EXTERNAL_TEST").sum()
    == 30,
    "1-30 June 2026",
)
add_check(
    "development_model_ready_rows_144",
    (
        assignment["evaluation_block"].eq(
            "DEVELOPMENT"
        )
        & assignment["model_ready"]
    ).sum()
    == 144,
    "1,584 contract cells",
)
add_check(
    "holdout_model_ready_rows_40",
    (
        assignment["evaluation_block"].eq(
            "INTERNAL_HOLDOUT"
        )
        & assignment["model_ready"]
    ).sum()
    == 40,
    "440 contract cells",
)
add_check(
    "external_model_ready_rows_119",
    (
        assignment["evaluation_block"].eq(
            "EXTERNAL_TEST"
        )
        & assignment["model_ready"]
    ).sum()
    == 119,
    "1,309 contract cells",
)
add_check(
    "four_expanding_validation_folds",
    set(
        assignment.loc[
            assignment["evaluation_block"].eq(
                "DEVELOPMENT"
            ),
            "development_fold",
        ].dropna().astype(int)
    )
    == {1, 2, 3, 4},
    "fold sizes 10,10,9,9 dates",
)
add_check(
    "all_dates_keep_all_rules_together",
    assignment.groupby(
        "event_date"
    )["evaluation_block"].nunique().eq(1).all(),
    "date-level partition",
)
add_check(
    "all_development_dates_keep_one_fold",
    assignment.loc[
        assignment["evaluation_block"].eq(
            "DEVELOPMENT"
        )
    ].groupby(
        "event_date"
    )["development_fold"].nunique().eq(1).all(),
    "date-level fold assignment",
)
add_check(
    "no_market_information_used",
    not assignment[
        "market_information_required_for_split"
    ].any(),
    "split uses chronology, weather availability and history",
)
add_check(
    "no_probability_bridge_used",
    not assignment[
        "probability_bridge_required_for_split"
    ].any(),
    "split construction is distribution-free",
)
add_check(
    "model_specification_not_selected",
    not assignment[
        "model_specification_selected"
    ].any(),
    "selection deferred",
)

integrity = pd.DataFrame(check_rows)
if not integrity["passed"].all():
    raise AssertionError(
        "18uA blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "event_date",
        "decision_rule",
        "detail",
        "blocking",
    ]
)

print("18uA integrity checks: PASS")

18uA integrity checks: PASS


In [6]:
output_frames = {
    "date_partition": date_partition,
    "date_rule_assignment": assignment,
    "block_summary": block_summary,
    "fold_summary": fold_summary,
    "fold_rule_summary": fold_rule_summary,
    "integrity": integrity,
    "issues": issues,
}

output_paths = {
    "date_partition": (
        OUT_DIR / "18uA_date_partition.csv"
    ),
    "date_rule_assignment": (
        OUT_DIR / "18uA_date_rule_assignment.csv"
    ),
    "block_summary": (
        OUT_DIR / "18uA_block_summary.csv"
    ),
    "fold_summary": (
        OUT_DIR / "18uA_fold_summary.csv"
    ),
    "fold_rule_summary": (
        OUT_DIR / "18uA_fold_rule_summary.csv"
    ),
    "integrity": (
        OUT_DIR / "18uA_integrity_checks.csv"
    ),
    "issues": OUT_DIR / "18uA_issues.csv",
}

for key, frame in output_frames.items():
    output = frame.copy()

    for column in output.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                output[column]
            ):
                output[column] = output[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "cutoff" in column.lower()
            or column.lower().endswith("_utc")
            or column.lower().endswith("_hkt")
            or "available" in column.lower()
        ):
            output[column] = output[column].astype(
                "string"
            )

    output.to_csv(
        output_paths[key],
        index=False,
    )

protocol = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "minimum_same_rule_history": MIN_HISTORY,
    "contracts_per_date_rule_book": CONTRACTS_PER_BOOK,
    "date_blocks": {
        "history_only_warmup": {
            "start": str(
                date_partition["event_date"].min().date()
            ),
            "end": str(WARMUP_END.date()),
            "dates": 25,
        },
        "development": {
            "start": str(DEVELOPMENT_START.date()),
            "end": str(DEVELOPMENT_END.date()),
            "dates": 38,
            "date_rule_rows": 152,
            "model_ready_date_rule_rows": 144,
            "model_ready_contract_cells": 1584,
        },
        "internal_holdout": {
            "start": str(HOLDOUT_START.date()),
            "end": str(HOLDOUT_END.date()),
            "dates": 10,
            "model_ready_date_rule_rows": 40,
            "model_ready_contract_cells": 440,
        },
        "external_test": {
            "start": str(EXTERNAL_START.date()),
            "end": str(EXTERNAL_END.date()),
            "dates": 30,
            "model_ready_date_rule_rows": 119,
            "model_ready_contract_cells": 1309,
            "known_missing_current_path": {
                "event_date": "2026-06-24",
                "decision_rule": "6h_prior",
            },
        },
    },
    "folds": [
        {
            "fold": int(row.development_fold),
            "validation_start": str(
                pd.Timestamp(
                    row.validation_start_date
                ).date()
            ),
            "validation_end": str(
                pd.Timestamp(
                    row.validation_end_date
                ).date()
            ),
            "validation_dates": int(
                row.validation_dates
            ),
            "validation_date_rule_rows": int(
                row.validation_date_rule_rows
            ),
            "validation_contract_cells": int(
                row.validation_contract_cells
            ),
        }
        for row in fold_summary.itertuples(index=False)
    ],
    "all_contracts_from_one_date_share_partition": True,
    "all_contracts_from_one_date_share_fold": True,
    "holdout_used_for_model_selection": False,
    "external_test_used_for_model_selection": False,
    "model_specification_selected": False,
    "selection_score_hierarchy_selected": False,
    "calibrator_selected": False,
    "trading_threshold_selected": False,
    "final_modelling_split_assigned": True,
    "probability_bridge_retained": False,
    "market_information_used_for_split": False,
}

protocol_path = OUT_DIR / "18uA_protocol.json"
protocol_path.write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18tA_availability_panel",
            "path": str(
                AVAILABILITY_PATH.relative_to(ROOT)
            ),
            "rows": len(availability),
            "sha256": sha256_file(
                AVAILABILITY_PATH
            ),
        },
        {
            "input_role": "18tA_summary",
            "path": str(
                A_SUMMARY_PATH.relative_to(ROOT)
            ),
            "rows": 1,
            "sha256": sha256_file(A_SUMMARY_PATH),
        },
        {
            "input_role": "18tB_decision_inventory",
            "path": str(
                DECISION_PATH.relative_to(ROOT)
            ),
            "rows": len(decisions),
            "sha256": sha256_file(DECISION_PATH),
        },
        {
            "input_role": "18tB_information_set_summary",
            "path": str(
                INFORMATION_PATH.relative_to(ROOT)
            ),
            "rows": len(information),
            "sha256": sha256_file(INFORMATION_PATH),
        },
        {
            "input_role": "18tB_summary",
            "path": str(
                B_SUMMARY_PATH.relative_to(ROOT)
            ),
            "rows": 1,
            "sha256": sha256_file(B_SUMMARY_PATH),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18uA_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "settlement_dates": int(len(date_partition)),
    "date_rule_rows": int(len(assignment)),
    "minimum_same_rule_history": MIN_HISTORY,
    "history_only_warmup_dates": 25,
    "development_dates": 38,
    "development_model_ready_date_rule_rows": 144,
    "development_model_ready_contract_cells": 1584,
    "validation_folds": 4,
    "fold_validation_date_counts": [10, 10, 9, 9],
    "fold_validation_date_rule_rows": [32, 40, 36, 36],
    "internal_holdout_dates": 10,
    "internal_holdout_model_ready_date_rule_rows": 40,
    "internal_holdout_contract_cells": 440,
    "external_test_dates": 30,
    "external_model_ready_date_rule_rows": 119,
    "external_contract_cells": 1309,
    "all_contracts_from_one_date_share_partition": True,
    "all_contracts_from_one_date_share_fold": True,
    "model_specification_selected": False,
    "final_modelling_split_assigned": True,
    "probability_bridge_retained": False,
    "market_information_used_for_split": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(len(integrity)),
}

summary_path = OUT_DIR / "18uA_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "revision": "v1",
}
environment_path = OUT_DIR / "18uA_environment.json"
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18uA chronological partition and folds",
    "",
    "**PASS**",
    "",
    "## Frozen date blocks",
    "",
    "| Block | Dates | Date-rule rows | Model-ready rows | Contract cells |",
    "|---|---:|---:|---:|---:|",
]

for row in block_summary.itertuples(index=False):
    report_lines.append(
        f"| {row.evaluation_block} | "
        f"{int(row.settlement_dates)} | "
        f"{int(row.date_rule_rows)} | "
        f"{int(row.model_ready_rows)} | "
        f"{int(row.planned_contract_cells)} |"
    )

report_lines.extend(
    [
        "",
        "## Development folds",
        "",
        "| Fold | Start | End | Dates | Date-rule rows | Contract cells |",
        "|---:|---|---|---:|---:|---:|",
    ]
)

for row in fold_summary.itertuples(index=False):
    report_lines.append(
        f"| {int(row.development_fold)} | "
        f"{pd.Timestamp(row.validation_start_date).date()} | "
        f"{pd.Timestamp(row.validation_end_date).date()} | "
        f"{int(row.validation_dates)} | "
        f"{int(row.validation_date_rule_rows)} | "
        f"{int(row.validation_contract_cells)} |"
    )

report_lines.extend(
    [
        "",
        "## Frozen boundary",
        "",
        (
            "The minimum same-rule history is sixteen. The "
            "internal holdout is 22-31 May and the external "
            "test is 1-30 June. Neither block may be used for "
            "model, calibrator or threshold selection."
        ),
        "",
        (
            "Model family, score hierarchy, GP kernel, tree "
            "specification and calibration variant remain "
            "unselected."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18uA_chronological_partition_and_folds_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18uA_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = OUT_DIR / "18uA_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18uA chronological partition release: PASS")

{
  "step": "18uA",
  "generated_at_utc": "2026-07-21T21:54:36.474291+00:00",
  "verdict": "PASS",
  "settlement_dates": 103,
  "date_rule_rows": 412,
  "minimum_same_rule_history": 16,
  "history_only_warmup_dates": 25,
  "development_dates": 38,
  "development_model_ready_date_rule_rows": 144,
  "development_model_ready_contract_cells": 1584,
  "validation_folds": 4,
  "fold_validation_date_counts": [
    10,
    10,
    9,
    9
  ],
  "fold_validation_date_rule_rows": [
    32,
    40,
    36,
    36
  ],
  "internal_holdout_dates": 10,
  "internal_holdout_model_ready_date_rule_rows": 40,
  "internal_holdout_contract_cells": 440,
  "external_test_dates": 30,
  "external_model_ready_date_rule_rows": 119,
  "external_contract_cells": 1309,
  "all_contracts_from_one_date_share_partition": true,
  "all_contracts_from_one_date_share_fold": true,
  "model_specification_selected": false,
  "final_modelling_split_assigned": true,
  "probability_bridge_retained": false,
  "market_informatio